In [1]:
import pandas as pd
import numpy as np
from src.features.process_race_infos import preprocess_pipeline_race_info
from src.features.process_features import *

/Users/axelcebille/Documents/GreyhoundRacingPrediction/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 0. Load datasets

In [2]:
full_dog_infos = pd.read_csv("../data/intermediate/08_all_race_infos.csv")
full_race_header = pd.read_csv("../data/intermediate/07_all_race_header.csv")
processed_race_header = pd.read_csv("../data/processed/09_all_race_header.csv")
all_past_dog_infos = pd.read_csv("../data/processed/05_dog_infos_engineered.csv")

In [3]:
full_race_header["raceDate"] = pd.to_datetime(full_race_header["raceDate"], errors="coerce")

/var/folders/m_/00q0qn914cs98zd_8p4zlxmc0000gn/T/ipykernel_22629/2201839702.py:1: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  full_race_header["raceDate"] = pd.to_datetime(full_race_header["raceDate"], errors="coerce")


# 1. Preliminary cleaning

We start by processing the dog infos to handle missing values or anomalies.

## 1.1 Remove "No Race" occurences

Here, we remove races that did not happen at all.

In [5]:
full_dog_infos = full_dog_infos[(~full_dog_infos["resultComment"].str.contains("(NoRace)", na=False))]
full_dog_infos.reset_index(inplace=True)

/var/folders/m_/00q0qn914cs98zd_8p4zlxmc0000gn/T/ipykernel_22629/4215121099.py:1: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  full_dog_infos = full_dog_infos[(~full_dog_infos["resultComment"].str.contains("(NoRace)", na=False))]


## 1.2 Non-racing dogs in happened races

To handle dogs that did not race but where supposed to (injuries, other issues), we assign them a result position of 8 and a flag `didNotRace`. 

This allows us to not ignore all the races where not 6 dogs runned and still get informations from them.

In [6]:
full_dog_infos.fillna({"resultPosition": 0}, inplace=True)
full_dog_infos['didNotRace'] = (full_dog_infos['resultPosition'] == 0).astype(int)
full_dog_infos['resultPosition'] = full_dog_infos['resultPosition'].replace(0, 8)

## 1.3 Put dog infos in trap order

The data is organized so that the dogs are put in the order they finished. This is a bad thing if we feed it to the model like this. It will always predict that the first dog in the dataset is always winning. So, we order by `trapNumber`.

In [7]:
full_dog_infos.sort_values(by=["source_file", "trapNumber"],inplace=True)

##  ## Files names without exactly 6 dogs

In [8]:
counts = full_dog_infos["source_file"].groupby(full_dog_infos["source_file"]).count()

In [9]:
counts[counts != 6]

source_file
1001838.csv    1
1021242.csv    1
1026222.csv    1
1032933.csv    1
1038372.csv    1
              ..
978237.csv     1
980640.csv     1
983577.csv     1
984410.csv     1
988882.csv     1
Name: source_file, Length: 95, dtype: int64

#################

# 2. Feature engineering

## 2.1 Early feature engineering

We start with feature engineering that do not rely on the past dog race infos. 

We define:

- `dogSex`: flag for the model to handle dog sex

- `dogAge`: age of the dog the day of the race


In [10]:
# merge on source_file to get the race date for each dog info
full_dog_infos = full_dog_infos.merge(full_race_header[["raceDate","source_file"]], on=["source_file"])

In [11]:
full_dog_infos["dogSex"] = full_dog_infos["dogSex"].fillna("b")
full_dog_infos["dogSex"] = pd.Categorical(full_dog_infos["dogSex"], categories=["b","d"])

full_dog_infos["raceDate"] = pd.to_datetime(full_dog_infos["raceDate"], format="%d/%m/%Y")
full_dog_infos["dogBorn"] = pd.to_datetime(full_dog_infos["dogBorn"], format="%b-%Y")
#full_dog_infos["dogAge"] = full_dog_infos["dogBorn"].apply(lambda x: calculate_dog_age(full_dog_infos["raceDate"], x)) 

full_dog_infos["dogAge"] = (
    full_dog_infos["raceDate"] - full_dog_infos["dogBorn"]
).dt.days / 365.25

full_dog_infos.drop(columns=["dogBorn"], inplace=True)

## 2.2 Past dog infos feature engineering

In [12]:
def find_race_info_from_header(file_name, full_race_header, full_dog_infos):
    return None

In [13]:
def compute_n_averages_stats(dog_infos, n=5, relevance_weighted=True):
    """
    Compute the averages of the last n races for the given dog_infos DataFrame. 
    """
    last_n = dog_infos.iloc[:n].copy()  # get last n races
    if last_n.shape[0] == 0:
        avg_stats = pd.Series(
            [np.nan] * len(dog_infos.columns),
            index=dog_infos.columns
        )
    else:
        # Coerce to numeric and sanitize infinities to avoid invalid reductions
        last_n_numeric = last_n.apply(pd.to_numeric, errors="coerce")
        last_n_numeric = last_n_numeric.replace([np.inf, -np.inf], np.nan)

        if relevance_weighted:
            weights = pd.to_numeric(
                last_n["relevance"],
                errors="coerce"
            ).fillna(0.0)
            denom = weights.sum()
            if not np.isfinite(denom) or denom == 0:
                weights = pd.Series(
                    np.ones(len(last_n)) / len(last_n),
                    index=last_n.index
                )
            else:
                weights = weights / denom
        else:
            weights = pd.Series(
                np.ones(len(last_n)) / len(last_n),
                index=last_n.index
            )

        avg_stats = (last_n_numeric * weights.values[:, np.newaxis]).sum()

    avg_stats_row = pd.DataFrame(avg_stats.values.reshape(1, -1), columns=avg_stats.index)
    if relevance_weighted:
        avg_stats_row.columns = [f"Weighted{col}" + f"{n}" for col in avg_stats_row.columns]
    else:
        avg_stats_row.columns = [col + f"{n}" for col in avg_stats_row.columns]

    return avg_stats_row

In [14]:
def compute_dog_average_stats(dog_id, all_past_dog_infos, race_date):
    # Filter the dog_infos for the given dog_id and only keep races before the given race_date
    dog_infos = all_past_dog_infos[all_past_dog_infos["dogId"] == dog_id].copy()
    race_date = pd.to_datetime(race_date, errors="coerce")
    dog_infos["raceDate"] = pd.to_datetime(dog_infos["raceDate"], errors="coerce")
    dog_infos = dog_infos[dog_infos["raceDate"] < race_date].reset_index(drop=True)
    ##########################################################################################

    cols_to_drop = ["resultComment","raceTime","raceId","raceType","raceClass","meetingId",
                "trackName","dogId","resultSectionalTime_missing","classGroup","inLSTM","raceClassFamily"]
    dog_infos = dog_infos.drop(columns=cols_to_drop)

    # Compute the average stats for the last n races and concat together
    avg_stats_last3 = compute_n_averages_stats(dog_infos, n=3, relevance_weighted=True)
    avg_stats7 = compute_n_averages_stats(dog_infos, n=7, relevance_weighted=True)
    avg_stats = compute_n_averages_stats(dog_infos, n=200, relevance_weighted=True)

    full_stats = pd.concat([avg_stats_last3, avg_stats7, avg_stats], axis=1)
    ##########################################################################################

    for n in [3,7,200]:
        full_stats[f"winPercentage{n}"] = compute_win_percentage(dog_infos, n=n)
        full_stats[f"oneTwoPercentage{n}"] = compute_one_two_percentage(dog_infos, n=n)
        full_stats[f"showPercentage{n}"] = compute_show_percentage(dog_infos, n=n)

    last_race_date = dog_infos["raceDate"].iloc[0]
    full_stats["DaysSinceLastRace"] = (race_date - last_race_date).days

    full_stats["num_races"] = len(dog_infos)
    full_stats.drop(columns=["WeightedraceDate3","WeightedraceDate7","WeightedraceDate200"], inplace=True)
    return full_stats

In [15]:
useful_cols = ["trapNumber","dogId","resultPosition","source_file","didNotRace","dogAge"]
full_dog_infos = full_dog_infos[useful_cols]

## PIPELINE TO ADAPT:

In [16]:
file_example = "1.csv"
race_info_example = full_dog_infos[full_dog_infos["source_file"] == file_example]
race_header_example = full_race_header[full_race_header["source_file"] == file_example]

raceDate = race_header_example["raceDate"].iloc[0]

full_stats = pd.DataFrame()
for i in range(6):
    dogID_i = race_info_example["dogId"].iloc[i]
    stats = compute_dog_average_stats(dogID_i, all_past_dog_infos, raceDate)

    full_stats = pd.concat([full_stats, stats], axis=0)

full_stats.reset_index(inplace=True, drop=True)
final_stats = pd.concat([race_info_example.reset_index(drop=True), full_stats], axis=1)